In [1]:
import os
import cv2
import numpy as np
from sklearn.neighbors import NearestNeighbors
import imgaug.augmenters as iaa
from PIL import Image

# Load image and resize
def load_and_resize_image(image_path, size=(64, 64)):
    img = Image.open(image_path)
    img = img.resize(size)
    return np.array(img).flatten()

# Extract features from images for k-NN
def extract_image_features(folder):
    image_files = os.listdir(folder)
    images = [load_and_resize_image(os.path.join(folder, image)) for image in image_files]
    return np.array(images), image_files

# k-NN based SMOTE augmentation
def smote_augment_images(folder, target_count, k=5):
    images, image_files = extract_image_features(folder)
    
    # Fit k-NN to find neighbors
    nbrs = NearestNeighbors(n_neighbors=k, algorithm='auto').fit(images)
    _, indices = nbrs.kneighbors(images)
    
    augment = iaa.Sequential([iaa.Affine(rotate=(-20, 20)), iaa.Fliplr(1.0)])
    
    # Augment and save new images
    while len(os.listdir(folder)) < target_count:
        random_image_idx = np.random.randint(0, len(images))
        neighbor_idx = np.random.choice(indices[random_image_idx])
        
        img1_path = os.path.join(folder, image_files[random_image_idx])
        img2_path = os.path.join(folder, image_files[neighbor_idx])
        
        img1 = Image.open(img1_path)
        img2 = Image.open(img2_path)
        
        # Blend two neighbor images to simulate SMOTE
        blended_img = Image.blend(img1, img2, alpha=0.5)
        augmented_img = augment(image=np.array(blended_img))
        
        augmented_img_pil = Image.fromarray(augmented_img)
        new_image_name = f"smote_{len(os.listdir(folder))}.png"
        augmented_img_pil.save(os.path.join(folder, new_image_name))

# Example usage
cat_dir = r'B:\Python_projects\Skin Cancer\New_data\binary_class-balanced\train\benign'
dog_dir = r'B:\Python_projects\Skin Cancer\New_data\binary_class-balanced\train\malignant'

# Balance classes
if len(os.listdir(cat_dir)) < len(os.listdir(dog_dir)):
    smote_augment_images(cat_dir, len(os.listdir(dog_dir)))
else:
    smote_augment_images(dog_dir, len(os.listdir(cat_dir)))